# Module 3: Student Lab

Three exercises. The first two are core; the third is an optional capstone.
Each has a test cell that prints `PASS` when your code is correct.

In [ ]:
import sys, os
# Make the engine importable whether this notebook is launched from the
# repo root or from the education/ folder.
for candidate in ('.', '..'):
    if os.path.exists(os.path.join(candidate, 'workflow_engine.py')):
        sys.path.insert(0, os.path.abspath(candidate))
        break

from workflow_engine import WorkflowEngine
from base_node import BaseNode
print('ParcelFlow loaded.')

## Exercise 1: The String Splitter

Implement `SplitterNode`: it requires a parcel `sentence` and produces a parcel
`words` containing the list of words (split on spaces).

Fill in `run`, then run the test cell.

In [ ]:
class SplitterNode(BaseNode):
    def __init__(self):
        super().__init__('splitter', requires=['sentence'], outputs=['words'])

    def run(self, parcels, index=None):
        # TODO: read parcels['sentence'].value, split on spaces,
        # and return {'words': <list of words>}
        pass

In [ ]:
# TEST CELL - do not edit. Prints PASS when Exercise 1 is correct.
engine = WorkflowEngine()
result = engine.execute_workflow([SplitterNode()], {'sentence': 'hello world from parcelflow'})
assert 'words' in result, 'SplitterNode produced no \'words\' parcel'
assert result['words'].value == ['hello', 'world', 'from', 'parcelflow'], result['words'].value
print('PASS')

## Exercise 2: Diagnose and fix a deadlock

The workflow below never finishes its second step. `StartNode` produces a parcel
called `step1`, but `BadNode` requires `step_one`. Because no node produces
`step_one`, `BadNode` never becomes ready.

Run the cell and read the engine's report at the end of the log.
Then fix `BadNode`'s `requires` so the workflow completes.

In [ ]:
class StartNode(BaseNode):
    def __init__(self):
        super().__init__('start', requires=['input'], outputs=['step1'])
    def run(self, parcels, index=None):
        return {'step1': 'done'}

class BadNode(BaseNode):
    def __init__(self):
        # BUG: this requires a parcel name no node produces. Fix it.
        super().__init__('bad', requires=['step_one'], outputs=['final'])
    def run(self, parcels, index=None):
        return {'final': 'success'}

engine = WorkflowEngine()
engine.execute_workflow([StartNode(), BadNode()], {'input': 'go'})

In [ ]:
# TEST CELL - do not edit. Prints PASS once BadNode is fixed.
engine = WorkflowEngine()
result = engine.execute_workflow([StartNode(), BadNode()], {'input': 'go'})
assert 'final' in result, 'BadNode still never runs - check its requires list'
assert engine.unrun_nodes == [], 'Some node still deadlocked'
print('PASS')

## Exercise 3 (capstone): make the independent work concurrent

The engine runs sequentially by design. But when a list is scattered into items,
each per-item computation is **independent** — and independent work can run
concurrently. That independence is what real systems exploit; here you will
exploit it directly.

Below, `slow_process` sleeps to imitate real work. Use
`concurrent.futures.ThreadPoolExecutor` to process all items concurrently. The
results must match the sequential version; the concurrent version should be
noticeably faster.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def slow_process(item):
    time.sleep(0.2)  # imitate real per-item work
    return item.upper()

items = ['alice', 'bob', 'charlie', 'diana', 'erin']

# Baseline: sequential
t0 = time.time()
sequential = [slow_process(x) for x in items]
seq_time = time.time() - t0
print(f'sequential: {sequential}  ({seq_time:.2f}s)')

def process_concurrently(items):
    # TODO: use ThreadPoolExecutor to run slow_process over all items
    # concurrently, returning a list of results in the original order.
    pass

In [ ]:
# TEST CELL - do not edit. Prints PASS when Exercise 3 is correct.
t0 = time.time()
concurrent_result = process_concurrently(items)
conc_time = time.time() - t0
print(f'concurrent: {concurrent_result}  ({conc_time:.2f}s)')
assert concurrent_result == sequential, 'Results must match the sequential version (and keep order)'
assert conc_time < seq_time, 'Concurrent version should be faster than sequential'
print('PASS')

When you finish, look back at `workflows.py`: the engine itself still runs nodes
one at a time. You have just shown why it *could* run the independent ones at
once — which is exactly the distinction between the *structure* of parallel work
and its *execution*.